In [1]:
# ==================================================================================
# REFACTORED EEG-TO-TEXT MODEL TRAINING SCRIPT (V3)
#
# KEY ARCHITECTURAL CHANGES:
# 1. NEW: AttentionMetaHead - Attends over encoder_outputs instead of using only
#    the final global_eeg_context
# 2. MODIFIED: Decoder no longer receives global_eeg_context at each step
#    (only uses it for initialization)
# 3. MODIFIED: Seq2Seq forward pass updated to use the new meta-head
# 4. CRITICAL: 'evaluate' function uses meta_teacher_forcing_ratio=0.0
#    for an HONEST validation score.
# ==================================================================================

import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from statsmodels.tsa.stattools import grangercausalitytests
from torch_geometric.utils import from_scipy_sparse_matrix, add_self_loops
from scipy.sparse import coo_matrix
import time
import math
import random
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import json
import evaluate as hf_evaluate # Use alias

/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


In [2]:
# ==================================================================================
# --- Focal Loss Setup (Using BCE Fallback) ---
# We discovered the focal-loss packages are for multi-class, not multi-label.
# BCEWithLogitsLoss is the correct loss function for our multi-label object task.
# ==================================================================================
print("Using nn.BCEWithLogitsLoss for object loss (correct for multi-label).")
focal_loss_criterion = nn.BCEWithLogitsLoss()

Using nn.BCEWithLogitsLoss for object loss (correct for multi-label).


In [3]:
# ==================================================================================
# CONSTANTS
# ==================================================================================
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
TRAIN_PCT, VAL_PCT = 0.8, 0.1
BATCH_SIZE = 16

NUM_COLORS = 12
NUM_OBJECTS = 90

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Metadata config: {NUM_COLORS} colors, {NUM_OBJECTS} objects (NO categories)")

Using device: cuda
Metadata config: 12 colors, 90 objects (NO categories)


In [4]:
# ==================================================================================
# DATASET AND DATALOADER
# ==================================================================================
class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')

        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype(np.int64))

        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)

    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)

    return eeg_batch.float(), meta_batch.float(), text_padded

In [5]:
# ==================================================================================
# GRANGER CAUSALITY
# ==================================================================================
def create_granger_causality_matrix(eeg_batch):
    eeg_sample = eeg_batch[0].cpu().numpy().T
    num_channels = eeg_sample.shape[1]
    causality_matrix = np.zeros((num_channels, num_channels))

    for i in range(num_channels):
        for j in range(num_channels):
            if i == j:
                continue
            ts_i = eeg_sample[:, i]
            ts_j = eeg_sample[:, j]
            min_len = 20
            if len(ts_i) < min_len or len(ts_j) < min_len:
                causality_matrix[i, j] = 0.0
                continue
            data = np.vstack([ts_j, ts_i]).T
            try:
                current_maxlag = min(5, len(data)//2 - 2)
                if current_maxlag < 1:
                    causality_matrix[i, j] = 0.0
                    continue
                results = grangercausalitytests(data, maxlag=current_maxlag, verbose=False)
                p_value = results[current_maxlag][0]['ssr_ftest'][1]
                if p_value < 0.05:
                    causality_matrix[i, j] = 1.0
            except Exception as e:
                causality_matrix[i, j] = 0.0

    adj_matrix = coo_matrix(causality_matrix)
    edge_index, edge_attr = from_scipy_sparse_matrix(adj_matrix)

    if edge_attr is None:
        edge_attr = torch.tensor([], dtype=torch.float)
    elif edge_attr.ndim == 0:
        edge_attr = edge_attr.unsqueeze(0)

    return edge_index.to(torch.long), edge_attr.to(torch.float)

In [6]:
# ==================================================================================
# NEW: ATTENTION-BASED META-HEAD
# ==================================================================================
class AttentionMetaHead(nn.Module):
    """
    Predicts metadata (color + objects) using attention over encoder outputs.
    """
    def __init__(self, enc_dim, hidden_dim, num_colors, num_objects, dropout=0.3):
        super().__init__()
        self.enc_dim = enc_dim
        self.hidden_dim = hidden_dim
        self.num_colors = num_colors
        self.num_objects = num_objects
        
        # Separate query generators for color and objects
        self.color_query_generator = nn.Sequential(
            nn.Linear(enc_dim, hidden_dim),
            nn.Tanh()
        )
        
        self.object_query_generator = nn.Sequential(
            nn.Linear(enc_dim, hidden_dim),
            nn.Tanh()
        )
        
        # Key/Value projections for encoder outputs
        self.key_projection = nn.Linear(enc_dim, hidden_dim)
        self.value_projection = nn.Linear(enc_dim, hidden_dim)
        
        # Final prediction heads
        self.color_predictor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_colors)
        )
        
        self.object_predictor = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_objects)
        )
        
        print(f"AttentionMetaHead initialized:")
        print(f"  - Input dim: {enc_dim}, Hidden dim: {hidden_dim}")
        print(f"  - Outputs: {num_colors} colors, {num_objects} objects")
    
    def attend(self, query, encoder_outputs):
        """
        Compute attention weights and context vector.
        
        Args:
            query: [batch, hidden_dim] - what to look for
            encoder_outputs: [seq_len, batch, enc_dim] - where to look
        """
        # [seq_len, batch, enc_dim] -> [seq_len, batch, hidden_dim]
        keys = self.key_projection(encoder_outputs)
        values = self.value_projection(encoder_outputs)
        
        # query: [batch, hidden_dim] -> [batch, 1, hidden_dim]
        # keys: [seq_len, batch, hidden_dim] -> [batch, seq_len, hidden_dim]
        query = query.unsqueeze(1)    # [batch, 1, hidden_dim]
        keys = keys.permute(1, 0, 2)  # [batch, seq_len, hidden_dim]
        
        # Scaled dot-product attention
        scores = torch.bmm(query, keys.transpose(1, 2))  # [batch, 1, seq_len]
        scores = scores / (self.hidden_dim ** 0.5)  # Scale
        
        attn_weights = F.softmax(scores, dim=-1)  # [batch, 1, seq_len]
        
        values = values.permute(1, 0, 2)  # [batch, seq_len, hidden_dim]
        context = torch.bmm(attn_weights, values)  # [batch, 1, hidden_dim]
        context = context.squeeze(1)  # [batch, hidden_dim]
        
        return context, attn_weights.squeeze(1)
    
    def forward(self, encoder_outputs, global_eeg_context):
        """
        Args:
            encoder_outputs: [seq_len, batch, enc_dim] - per-timestep features
            global_eeg_context: [batch, enc_dim] - final hidden state (used as initial query)
        """
        # Generate queries for each task from the global context
        color_query = self.color_query_generator(global_eeg_context)   # [batch, hidden_dim]
        object_query = self.object_query_generator(global_eeg_context)  # [batch, hidden_dim]
        
        # Attend to encoder outputs for each task
        color_context, color_attn_weights = self.attend(color_query, encoder_outputs)
        object_context, object_attn_weights = self.attend(object_query, encoder_outputs)
        
        # Predict metadata from attended contexts
        pred_color = self.color_predictor(color_context)    # [batch, num_colors]
        pred_object = self.object_predictor(object_context)  # [batch, num_objects]
        
        return pred_color, pred_object

In [7]:
# ==================================================================================
# ENCODER (Unchanged)
# ==================================================================================
class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers,
                          bidirectional=True, dropout=dropout if num_layers > 1 else 0,
                          batch_first=True)
        self.dropout = nn.Dropout(dropout)
        print(f"Encoder RNN input size: {enc_hidden}")

    def forward(self, eeg, edge_index, edge_attr):
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]

        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1])

        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)

        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))

        temporal_features = x.reshape(batch_size, num_timesteps, -1)
        encoder_outputs, encoder_hidden = self.rnn(temporal_features)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        return encoder_outputs, encoder_hidden

In [8]:
# ==================================================================================
# ATTENTION (Unchanged)
# ==================================================================================
class LuongAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim, dec_dim)

    def forward(self, decoder_hidden, encoder_outputs):
        src_len = encoder_outputs.shape[0]
        attn_energies = self.attn(encoder_outputs)
        scores = torch.bmm(decoder_hidden.permute(1, 0, 2), attn_energies.permute(1, 2, 0))
        attn_weights = F.softmax(scores, dim=2)
        context = torch.bmm(attn_weights, encoder_outputs.permute(1, 0, 2))
        return context, attn_weights.squeeze(1)

In [9]:
# ==================================================================================
# METADATA ENCODER (Unchanged)
# ==================================================================================
class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects, 
                 color_emb_dim=16, object_feature_dim=128):
        super().__init__()
        self.color_embedding = nn.Embedding(num_colors, color_emb_dim)
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, object_feature_dim)
        )
        self.output_dim = color_emb_dim + object_feature_dim
        print(f"MetadataEncoder output dimension: {self.output_dim}")

    def forward(self, metadata):
        color_ids = metadata[:, 0].long()
        object_features_raw = metadata[:, 1:].float()
        
        color_vec = self.color_embedding(color_ids)
        object_vec = self.object_processor(object_features_raw)
        
        combined_features = torch.cat([color_vec, object_vec], dim=1)
        return combined_features

In [10]:
# ==================================================================================
# MODIFIED DECODER - REMOVED global_eeg_context FROM PER-STEP INPUT
# ==================================================================================
class Decoder(nn.Module):
    """
    MODIFIED: No longer receives global_eeg_context at each decoding step.
    """
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, 
                 meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        enc_dim = enc_hidden * 2

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = LuongAttention(enc_dim, dec_hidden)

        # UPDATED: RNN input is now smaller (no global_eeg_context per step)
        self.rnn_input_dim = emb_dim + enc_dim + meta_features_dim
        print(f"Decoder RNN input dimension: {self.rnn_input_dim} (removed global context)")
        self.rnn = nn.GRU(self.rnn_input_dim, dec_hidden, num_layers, 
                          dropout=dropout if num_layers > 1 else 0)

        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_dim, dec_hidden)

    def init_hidden(self, encoder_hidden):
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last_layer_hidden = hidden[-1]
        encoder_hidden_cat = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        bridged_hidden = torch.tanh(self.bridge(encoder_hidden_cat))
        decoder_initial_hidden = bridged_hidden.unsqueeze(0).repeat(self.num_layers, 1, 1)
        return decoder_initial_hidden

    def forward(self, token, decoder_hidden, encoder_outputs, meta_features):
        """
        MODIFIED: Removed global_eeg_context parameter.
        """
        token = token.unsqueeze(0)
        embedded = self.dropout(self.embedding(token))
        context, attn_weights = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)
        
        meta_features_unsqueezed = meta_features.unsqueeze(0)
        context_permuted = context.permute(1, 0, 2)

        # UPDATED: Concatenate only 3 components
        rnn_input = torch.cat((
            embedded,
            context_permuted,
            meta_features_unsqueezed
        ), dim=2)

        output, hidden = self.rnn(rnn_input, decoder_hidden)
        prediction = self.fc_out(output.squeeze(0))

        return prediction, hidden, context.squeeze(1)

# ==================================================================================
# MODIFIED SEQ2SEQ - USES NEW ATTENTION META-HEAD AND UPDATED DECODER
# ==================================================================================
class Seq2Seq(nn.Module):
    """
    MAJOR CHANGES:
    1. Replaced simple MLP meta_head with AttentionMetaHead
    2. Updated forward pass to use attention-based metadata prediction
    3. Decoder no longer receives global_eeg_context at each step
    """
    def __init__(self, text_vocab_size, num_colors, num_objects, enc_hidden=256, dec_hidden=256,
                 pad_id=0, dropout=0.2, color_emb_dim=16, object_feature_dim=128, 
                 emb_dim=256, dec_layers=2, meta_hidden=256): # <-- Added meta_hidden
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, num_layers=dec_layers)
        
        self.meta_encoder = MetadataEncoder(
            num_colors, 
            num_objects,
            color_emb_dim, 
            object_feature_dim
        )

        meta_features_dim = self.meta_encoder.output_dim
        enc_dim = enc_hidden * 2

        # UPDATED: Use new Decoder
        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden,
                                 meta_features_dim, dec_layers, pad_id, dropout)

        # NEW: Attention-based meta-head instead of simple MLP
        self.meta_head = AttentionMetaHead(
            enc_dim=enc_dim,
            hidden_dim=meta_hidden,
            num_colors=num_colors,
            num_objects=num_objects,
            dropout=dropout
        )
        
        self.num_colors = num_colors
        self.num_objects = num_objects

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr, 
                text_teacher_forcing_ratio=0.5, meta_teacher_forcing_ratio=1.0):
        """
        UPDATED: 
        - Meta-head now uses attention over encoder_outputs
        - Decoder no longer receives global_eeg_context per step
        """
        batch_size = eeg.shape[0]
        target_len = target_text.shape[1]
        target_vocab_size = self.decoder.vocab_size

        # Encode EEG
        encoder_outputs, encoder_hidden = self.encoder(eeg, edge_index, edge_attr)
        
        # Get global EEG context (still needed for initialization and meta-head query)
        hidden_reshaped = encoder_hidden.view(self.encoder.rnn.num_layers, 2, batch_size, -1)
        last_layer_hidden = hidden_reshaped[-1]
        global_eeg_context = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)

        # UPDATED: Use attention-based meta-head
        pred_color, pred_object = self.meta_head(encoder_outputs, global_eeg_context)

        # Scheduled sampling for metadata (Unchanged from your previous version)
        use_true_meta = random.random() < meta_teacher_forcing_ratio
        
        if use_true_meta:
            meta_features = self.meta_encoder(metadata)
        else:
            with torch.no_grad():
                pred_color_id_vec = pred_color.argmax(dim=-1).float().unsqueeze(1)
                pred_object_vec = (torch.sigmoid(pred_object) > 0.5).float()
                predicted_meta_vector = torch.cat([pred_color_id_vec, pred_object_vec], dim=1)
            meta_features = self.meta_encoder(predicted_meta_vector)

        # Initialize decoder
        decoder_hidden = self.decoder.init_hidden(encoder_hidden)

        # Text generation loop
        outputs = torch.zeros(target_len, batch_size, target_vocab_size).to(eeg.device)
        decoder_input = target_text[:, 0]

        for t in range(1, target_len):
            # UPDATED: No longer pass global_eeg_context
            output, decoder_hidden, _ = self.decoder(
                decoder_input,
                decoder_hidden,
                encoder_outputs,
                meta_features
            )

            outputs[t] = output
            teacher_force = random.random() < text_teacher_forcing_ratio
            top1 = output.argmax(1)
            decoder_input = target_text[:, t] if teacher_force else top1

        return outputs[1:].permute(1, 0, 2), pred_color, pred_object

In [11]:
# ==================================================================================
# TRAINING FUNCTION (Unchanged from your V2)
# ==================================================================================
def train_one_epoch(model, loader, optimizer, text_criterion, color_criterion, object_criterion,
                    granger_edge_index, granger_edge_attr, 
                    color_loss_weight, object_loss_weight, 
                    meta_teacher_forcing_ratio=1.0):
    model.train()
    total_loss = 0.0
    total_loss_t, total_loss_c, total_loss_o = 0.0, 0.0, 0.0
    progress_bar = tqdm(loader, desc="Training", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)

        optimizer.zero_grad()

        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, 
            granger_edge_index, granger_edge_attr, 
            text_teacher_forcing_ratio=0.5,
            meta_teacher_forcing_ratio=meta_teacher_forcing_ratio
        )

        # Calculate losses
        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, 0].long())
        
        # Pass raw logits to BCEWithLogitsLoss
        loss_o = object_criterion(pred_object, meta_b[:, 1:].float())

        # Combined loss with new weights
        loss = loss_t + (color_loss_weight * loss_c) + (object_loss_weight * loss_o)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_loss_t += loss_t.item()
        total_loss_c += loss_c.item()
        total_loss_o += loss_o.item()

        progress_bar.set_postfix(
            loss=loss.item(), 
            txt=loss_t.item(), 
            clr=loss_c.item(), 
            obj=loss_o.item()
        )

    n = len(loader)
    return total_loss / n, total_loss_t / n, total_loss_c / n, total_loss_o / n

In [12]:
# ==================================================================================
# EVALUATION FUNCTION (The "Honest" one from your V2)
# ==================================================================================
@torch.no_grad()
def evaluate(model, loader, text_criterion, color_criterion, object_criterion,
             granger_edge_index, granger_edge_attr, 
             color_loss_weight, object_loss_weight):
    model.eval()
    total_loss = 0.0
    total_loss_t, total_loss_c, total_loss_o = 0.0, 0.0, 0.0
    progress_bar = tqdm(loader, desc="Evaluating", leave=False)

    for eeg_b, meta_b, txt_b in progress_bar:
        eeg_b, txt_b, meta_b = eeg_b.to(device), txt_b.to(device), meta_b.to(device)

        # CRITICAL: SET BOTH RATIOS TO 0.0 FOR TRUE, HONEST EVALUATION
        text_logits, pred_color, pred_object = model(
            eeg_b, meta_b, txt_b, 
            granger_edge_index, granger_edge_attr, 
            text_teacher_forcing_ratio=0.0,
            meta_teacher_forcing_ratio=0.0
        )

        loss_t = text_criterion(text_logits.reshape(-1, text_logits.shape[-1]), txt_b[:, 1:].reshape(-1))
        loss_c = color_criterion(pred_color, meta_b[:, 0].long())
        loss_o = object_criterion(pred_object, meta_b[:, 1:].float())

        loss = loss_t + (color_loss_weight * loss_c) + (object_loss_weight * loss_o)
        
        total_loss += loss.item()
        total_loss_t += loss_t.item()
        total_loss_c += loss_c.item()
        total_loss_o += loss_o.item()
        
        progress_bar.set_postfix(
            loss=loss.item(), 
            txt=loss_t.item(), 
            clr=loss_c.item(), 
            obj=loss_o.item()
        )

    n = len(loader)
    return total_loss / n, total_loss_t / n, total_loss_c / n, total_loss_o / n

In [14]:
# ==================================================================================
# MAIN EXECUTION BLOCK (Unchanged from your V2)
# ==================================================================================
if __name__ == "__main__":
    
    # --- Delete old checkpoint to ensure a fresh run ---
    # (This is good practice after a major architecture change)
    try:
        import os
        old_model_path = 'eeg-meta-text-qwen-refactored-model.pt'
        if os.path.exists(old_model_path):
            os.remove(old_model_path)
            print(f"Removed old checkpoint: '{old_model_path}'")
    except Exception as e:
        print(f"Could not remove old checkpoint: {e}")

    # Create dataset and loaders
    dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
    N = len(dataset)
    n_train = int(N * TRAIN_PCT)
    n_val = int(N * VAL_PCT)
    n_test = N - n_train - n_val
    g = torch.Generator().manual_seed(42)
    train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=g)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)

    # Create Granger matrix
    print("Creating Granger Causality matrix...")
    try:
        eeg_b, _, _ = next(iter(train_loader))
        granger_edge_index, granger_edge_attr = create_granger_causality_matrix(eeg_b)
        num_channels = eeg_b.shape[1]

        granger_edge_index, granger_edge_attr = add_self_loops(
            granger_edge_index,
            edge_attr=granger_edge_attr,
            num_nodes=num_channels,
            fill_value=1.0
        )

        if granger_edge_attr is None:
            granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float)

        granger_edge_index = granger_edge_index.to(torch.long).to(device)
        granger_edge_attr = granger_edge_attr.to(torch.float32).to(device)
        print(f"Granger matrix created: {granger_edge_index.shape}")

    except Exception as e:
        print(f"Error creating Granger matrix: {e}. Using fallback.")
        num_channels = 62
        edge_index = torch.combinations(torch.arange(num_channels), r=2).t().contiguous()
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        edge_index, _ = add_self_loops(edge_index, num_nodes=num_channels)
        granger_edge_index = edge_index.to(torch.long).to(device)
        granger_edge_attr = torch.ones(granger_edge_index.shape[1], dtype=torch.float32).to(device)

    # --- UPDATED MODEL INSTANTIATION ---
    model = Seq2Seq(
        text_vocab_size=TEXT_VOCAB_SIZE,
        num_colors=NUM_COLORS,
        num_objects=NUM_OBJECTS,
        pad_id=PAD_ID,
        dropout=0.2,
        enc_hidden=256,
        dec_hidden=256,
        emb_dim=256,
        dec_layers=2,
        meta_hidden=256  # <-- New parameter for the AttentionMetaHead
    ).to(device)

    print(f"Model instantiated on '{device}'.")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    # Setup losses
    object_criterion = focal_loss_criterion # This is our BCEWithLogitsLoss fallback
    text_criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    color_criterion = nn.CrossEntropyLoss()

    optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.2, patience=2, verbose=True)

    # Training loop
    EPOCHS = 40
    best_val_loss = float('inf')
    
    OBJECT_LOSS_WEIGHT = 50.0
    COLOR_LOSS_WEIGHT = 10.0

    meta_tf_schedule = np.linspace(1.0, 0.5, 20)

    print("\n--- Starting Training (V3 Architecture) ---")
    print(f"Object Loss Weight: {OBJECT_LOSS_WEIGHT} | Color Loss Weight: {COLOR_LOSS_WEIGHT}")
    print(f"Using 'Honest' Validation (meta_tf_ratio=0.0 in evaluate())")

    for epoch in range(1, EPOCHS + 1):
        start_time = time.time()

        if epoch - 1 < len(meta_tf_schedule):
            current_meta_tf_ratio = meta_tf_schedule[epoch - 1]
        else:
            current_meta_tf_ratio = meta_tf_schedule[-1]
            
        print(f"\nUsing Metadata Teacher Forcing Ratio (TRAIN): {current_meta_tf_ratio:.2f}")

        train_loss, tr_t, tr_c, tr_o = train_one_epoch(
            model, train_loader, optimizer,
            text_criterion, color_criterion, object_criterion,
            granger_edge_index, granger_edge_attr, 
            COLOR_LOSS_WEIGHT, OBJECT_LOSS_WEIGHT,
            meta_teacher_forcing_ratio=current_meta_tf_ratio
        )
        
        # This is the "Honest" validation
        val_loss, val_t, val_c, val_o = evaluate(
            model, val_loader,
            text_criterion, color_criterion, object_criterion,
            granger_edge_index, granger_edge_attr, 
            COLOR_LOSS_WEIGHT, OBJECT_LOSS_WEIGHT
        )

        scheduler.step(val_loss)
        end_time = time.time()
        epoch_mins = int((end_time - start_time) / 60)
        epoch_secs = int((end_time - start_time) % 60)

        print(f'\nEpoch: {epoch:02} | Time: {epoch_mins}m {epoch_secs}s')
        print(f'\tTrain Loss: {train_loss:.4f} | Txt: {tr_t:.4f} | Clr: {tr_c:.4f} | Obj: {tr_o:.4f}')
        print(f'\t  Val Loss: {val_loss:.4f} | Txt: {val_t:.4f} | Clr: {val_c:.4f} | Obj: {val_o:.4f} (HONEST)')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            model_save_path = 'eeg-meta-text-qwen-refactored-model.pt'
            torch.save(model.state_dict(), model_save_path)
            print(f"\t-> Val loss decreased. Saving best model to '{model_save_path}'")
        else:
            print("\t-> Val loss did not improve.")

    print("\n--- Training Complete ---")
    
    # (The inference part from your old script is removed, as it's
    # better to use the standalone 'run_inference.py' script)

Removed old checkpoint: 'eeg-meta-text-qwen-refactored-model.pt'
Creating Granger Causality matrix...
Granger matrix created: torch.Size([2, 3195])
Encoder RNN input size: 256
MetadataEncoder output dimension: 144
Decoder RNN input dimension: 912 (removed global context)
AttentionMetaHead initialized:
  - Input dim: 512, Hidden dim: 256
  - Outputs: 12 colors, 90 objects
Model instantiated on 'cuda'.
Total parameters: 20,008,416

--- Starting Training (V3 Architecture) ---
Object Loss Weight: 50.0 | Color Loss Weight: 10.0
Using 'Honest' Validation (meta_tf_ratio=0.0 in evaluate())

Using Metadata Teacher Forcing Ratio (TRAIN): 1.00


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/175 [00:00<?, ?it/s]


Epoch: 01 | Time: 7m 39s
	Train Loss: 35.8495 | Txt: 5.7977 | Clr: 2.2507 | Obj: 0.1509
	  Val Loss: 31.9547 | Txt: 4.9737 | Clr: 2.2186 | Obj: 0.0959 (HONEST)
	-> Val loss decreased. Saving best model to 'eeg-meta-text-qwen-refactored-model.pt'

Using Metadata Teacher Forcing Ratio (TRAIN): 0.97


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

...

Training:   0%|          | 0/1400 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/175 [00:00<?, ?it/s]


Epoch: 11 | Time: 7m 38s
	Train Loss: 30.0364 | Txt: 3.3512 | Clr: 2.1930 | Obj: 0.0951
	  Val Loss: 31.8662 | Txt: 5.0379 | Clr: 2.2125 | Obj: 0.0941 (HONEST)
	-> Val loss did not improve.

Using Metadata Teacher Forcing Ratio (TRAIN): 0.71


Training:   0%|          | 0/1400 [00:00<?, ?it/s]

KeyboardInterrupt: 